____
# BIOSWOT : Sensitivity to stain and vorticity


In [1]:
import numpy as np
import pandas as pd
import xarray as xr

import matplotlib.pyplot as plt

import os
from glob import glob

from swot import add_mask_inside_swot, build_swath_polygon
from cstes import swot_dir, drifters_dir, get_proj, lonlat2xy, zarr_dir, surface_drifters, depth_drifters, depth_100, depth_50, images_dir, U2, err_acc
from diagnosis import update_dict, dataset_coloc_combs, dataset_coloc_combs, prepared_drifters, prepared_wd, prepared_alti, compute_variance_groupby, compute_variance
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import cartopy.geodesic as cgeo
crs = ccrs.PlateCarree()

import cartopy.geodesic as geod
import cartopy.crs as ccrs
import cartopy.feature as cfeature

import pyproj
from pyproj import Geod

from rasterio.transform import Affine

import pynsitu as pyn

from diagnosis import synthetic_figure
from cstes import U2, c0

# Characteritics :
- from April 20th to 18th (cycle 497-515)
- 4-5°E; 40.3-41.3°N
- move from W to E
- size ?

In [1]:
def sel_bioswot(dfr):
    id = dfr[(dfr.longitude>4.75)&(dfr.longitude<5.5)&(dfr.latitude>40.3)&(dfr.latitude<41.3)&(dfr.cycle_number>498)&(dfr.cycle_number<523)].drifter_id.unique()
    
    dfr = dfr[((dfr.drifter_id.isin(id))
               &(dfr.longitude>4)
               &(dfr.longitude<5.5)
               &(dfr.latitude>40.3)
               &(dfr.latitude<41.3)
               &(dfr.cycle_number>498)
               &(dfr.cycle_number<523)
               &(~((dfr.longitude<4.75)&(dfr.latitude<40.6)))
              )]
    return dfr

def generate_0_15m_coloc_dataframe(freq_band):
    
    DT = 12
    
    # SURFACE
    base_dic = dict(dt='12h', 
            drifter_preprocess = 'spectral_decomp',
            drifter_preprocess_param= freq_band,
            alti_product_key='swot2km',
            alti_diff_method = 'fromduacsv',
            alti_diff_method_param = '',
            ggd_var = 'fromduacsv',
            wd_product_key = 'era5', 
            wd_model ='rio', 
            wd_depth = '0')
    
    
    comb_list = [base_dic]
    
    
    ds0 = dataset_coloc_combs(comb_list, nearest =False, remove_id_outliers = True)
    ds0 = ds0.where(ds0.depth==0, drop=True)
    #distance_to_coast = ds0.distance_to_coast.isel(id_comb=0)
    #ds0['distance_to_coast'] = ('row_number', distance_to_coast.values)
    #ds0 = ds0.where(ds0.time_to_swot/pd.Timedelta('1h')<DT, drop=True)
    #ds0 = ds0.where((abs(ds0.ggde)<1e-4) & (abs(ds0.ggdn)<1e-4), drop=True)
    #ds0 = ds0.where(ds0.distance_to_coast>distance_to_coast_max, drop=True)
    
    df0 = ds0.to_dataframe().reset_index().set_index('row_number')
    
    #DEPTH 15M
    base_dic = dict(dt='12h', 
                drifter_preprocess = 'spectral_decomp',
                drifter_preprocess_param=freq_band,
                alti_product_key='swot2km',
                alti_diff_method = 'fromduacsv',
                alti_diff_method_param = '',
                ggd_var = 'fromduacsv',
                wd_product_key = 'era5', 
                wd_model ='rio', 
                wd_depth = '15')
    
    comb_list = [    
        base_dic, 
    ]
    
    ds15 = dataset_coloc_combs(comb_list, nearest =False, remove_id_outliers = True)
    ds15 =ds15.where(ds15.depth==15, drop=True)
    #distance_to_coast = ds15.distance_to_coast.isel(id_comb=0)
    #ds15['distance_to_coast'] = ('row_number', distance_to_coast.values)
    ds15 = ds15.where(ds15.time_to_swot/pd.Timedelta('1h')<DT, drop=True)
    #ds15 = ds15.where((abs(ds15.ggde)<1e-4) & (abs(ds15.ggdn)<1e-4), drop=True)
    #ds15 = ds15.where(ds15.distance_to_coast>distance_to_coast_max, drop=True)
    
    df15 = ds15.to_dataframe().reset_index().set_index('row_number')

    return df0, df15

# Time evolution of the different components along trajectories

In [39]:
DF15 = dict()
DF0 = dict()

for spectral_band in ['', 'LF', 'inertial', 'diurnal', 'semidiurnal']:
    df0, df15 = generate_0_15m_coloc_dataframe(spectral_band)
    dfs = prepared_alti(dt='12h', 
            drifter_preprocess = '',
            drifter_preprocess_param='',
            alti_product_key='swot2km',
            alti_diff_method = 'xarray_diff' ,
            alti_diff_method_param = '')
    
    def concat(df, dfs):
        df = pd.concat([df, dfs[['vorticity_etaf', 'strain_s_etaf', 'strain_n_etaf',]]], axis=1).dropna()
        df['f'] =  2 * 2 * np.pi / 86164.1 * np.sin(df.latitude * np.pi / 180)
        fo = df.f.mean()
        df['strain_f'] = np.sqrt(df.strain_n_etaf**2 + df.strain_s_etaf**2)
        df['strain'] = df.strain_f*df.f#.assign_attrs({'long_name':r'$\sigma/|f|$'})
        df['vorticity'] = df.vorticity_etaf*df.f#.assign_attrs({'long_name':r'$\zeta/f$'})
        df['vorticity_f'] = df.vorticity_etaf#.assign_attrs({'long_name':r'$\zeta/f$'})
        df['omega'] = df.strain**2 - df.vorticity**2
        return df
    
    
    DF0[spectral_band] = sel_bioswot(concat(df0, dfs))
    DF15[spectral_band] = sel_bioswot(concat(df15, dfs))

321898
12h_spectral_decomp_all_med_variational_10min_v1____swot2km_fromduacsv__fromduacsv__era5_rio0
Identified SWOT and drifter outliers have been removed
321898
12h_spectral_decomp_all_med_variational_10min_v1____swot2km_fromduacsv__fromduacsv__era5_rio15
Identified SWOT and drifter outliers have been removed
321898
12h_spectral_decomp_all_med_variational_10min_v1__LF__swot2km_fromduacsv__fromduacsv__era5_rio0
Identified SWOT and drifter outliers have been removed
321898
12h_spectral_decomp_all_med_variational_10min_v1__LF__swot2km_fromduacsv__fromduacsv__era5_rio15
Identified SWOT and drifter outliers have been removed
321898
12h_spectral_decomp_all_med_variational_10min_v1__inertial__swot2km_fromduacsv__fromduacsv__era5_rio0
Identified SWOT and drifter outliers have been removed
321898
12h_spectral_decomp_all_med_variational_10min_v1__inertial__swot2km_fromduacsv__fromduacsv__era5_rio15
Identified SWOT and drifter outliers have been removed
321898
12h_spectral_decomp_all_med_variat

In [40]:
DF0['All'] = DF0['']
DF15['All'] = DF15['']

In [41]:
df = DF0['All']

,id_comb,sume,acce,exce_acc,core,exce_cor,ggde,exce_ggd,wde,exce_wd,...,distance_to_coast,vorticity_etaf,strain_s_etaf,strain_n_etaf,f,strain_f,strain,vorticity,vorticity_f,omega
row_number,,,,,,,,,,,,,,,,,,,,,
53515,12h_spectral_decomp_all_med_variational_10min_...,-4.010923e-07,0.000004,-0.000004,-0.000005,0.000005,0.000003,-3.103903e-06,-0.000002,0.000001,...,98618.999471,0.177633,-0.226923,-0.0,0.000096,0.226923,0.000022,0.000017,0.177633,1.823303e-10
53516,12h_spectral_decomp_all_med_variational_10min_...,6.547111e-07,0.000004,-0.000003,-0.000004,0.000005,0.000002,-1.822742e-06,-0.000002,0.000002,...,98602.178170,0.176036,-0.230096,-0.0,0.000096,0.230096,0.000022,0.000017,0.176036,2.007575e-10
53517,12h_spectral_decomp_all_med_variational_10min_...,1.363955e-06,0.000004,-0.000003,-0.000003,0.000005,0.000002,-9.118230e-07,-0.000002,0.000003,...,98674.151426,0.183303,-0.229102,-0.0,0.000096,0.229102,0.000022,0.000018,0.183303,1.727100e-10
53518,12h_spectral_decomp_all_med_variational_10min_...,1.934409e-06,0.000004,-0.000002,-0.000003,0.000005,0.000002,-3.614938e-08,-0.000002,0.000004,...,98780.448798,0.198898,-0.226662,-0.0,0.000096,0.226662,0.000022,0.000019,0.198898,1.080376e-10
53519,12h_spectral_decomp_all_med_variational_10min_...,2.593056e-06,0.000004,-0.000002,-0.000002,0.000004,0.000002,7.661096e-07,-0.000002,0.000004,...,98819.076702,0.217545,-0.219683,-0.0,0.000096,0.219683,0.000021,0.000021,0.217545,8.548538e-12
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
163808,12h_spectral_decomp_all_med_variational_10min_...,-1.299348e-05,-0.000007,-0.000006,0.000041,-0.000054,-0.000011,-2.083015e-06,-0.000036,0.000023,...,74707.423819,0.115870,-0.134303,-0.0,0.000095,0.134303,0.000013,0.000011,0.115870,4.124886e-11
163809,12h_spectral_decomp_all_med_variational_10min_...,-1.337949e-05,-0.000007,-0.000006,0.000042,-0.000055,-0.000011,-2.294160e-06,-0.000037,0.000023,...,74629.961174,0.110823,-0.132352,-0.0,0.000095,0.132352,0.000013,0.000010,0.110823,4.682701e-11
163810,12h_spectral_decomp_all_med_variational_10min_...,-1.388441e-05,-0.000008,-0.000006,0.000042,-0.000056,-0.000011,-2.505060e-06,-0.000037,0.000023,...,74518.734227,0.100240,-0.128639,-0.0,0.000095,0.128639,0.000012,0.000009,0.100240,5.813117e-11


In [38]:
ds['drifter_id'] = ds.drifter_id.astype(str)

def filter_traj(ds, T=20, cutoff = { 'HF':[2.2, 70.95], 'LF': [0.88],'inertial':[1.05, 1.75], 'diurnal' : [0.88, 1.05], 'semidiurnal':[1.9, 2.1],}):
    """Return dataset with filtered trajectories acceleration and coriolis term (inertial, diurnal, semidiurnal)
    T: window length days
    """
    from scipy import signal
    from scipy.signal import filtfilt
    from scipy.integrate import cumulative_trapezoid
    from scipy.optimize import minimize
    
    # coefficients
    dt = 1/24/6  # 10min in days
    numtaps = int(T / dt)
    pass_zero = False


    # filter
    dss = ds.fillna(0)
    for c in cutoff : 
        if c == 'LF' : pass_zero = True
        #elif c == 'HF' : pass_zero = 'highpass'
        else : pass_zero = False
 
        taps = signal.firwin(numtaps, cutoff=cutoff[c], pass_zero=pass_zero, fs=1 / dt, scale=True)
        dss[c+'_ggde'] = xr.DataArray(
            filtfilt(taps, 1, dss.ggde),
            dims=["drifter_id", "datetime"],
            )
        dss[c+'_ggdn'] = xr.DataArray(
            filtfilt(taps, 1, dss.ggdn),
            dims=["drifter_id","datetime"],
            )
        dss[c+'_wdn'] = xr.DataArray(
            filtfilt(taps, 1, dss.wdn),
            dims=["drifter_id","datetime"],
            )
        dss[c+'_wde'] = xr.DataArray(
            filtfilt(taps, 1, dss.wde),
            dims=["drifter_id","datetime"],
            )

        print(c)
    
    return dss


In [192]:
for i in range(len(df15.drifter_id.unique())):
    plot_timeseries(df15, i)
for i in range(len(df0.drifter_id.unique())):
    plot_timeseries(df0, i)

'/Users/mdemol/ownCloud/PhD/images/med_images'